<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Week 4 — How Sure Are We?</div>
</div>

#### Describing data, testing hypotheses and propagating uncertainty: Cavendish's density of the Earth, Michelson's speed of light, and a pendulum.
***

## Where this fits

Lecture 5 asked three questions about any result: what do the values look like, how uncertain is what we derive from them, and could it be chance? Today you answer them for two of the most famous measurements in physics, and for a pendulum.

1. Cavendish's 1798 density of the Earth: describe it, quote it, and test whether changing the apparatus mattered;
2. find glitches in a time-ordered series, on data where you know the answer;
3. Michelson's 1879 speed of light: a table of results, one row per experiment (and, optionally, random against systematic error);
4. *(optional)* propagate uncertainty into $g$, by the linear rule and by Monte Carlo;
5. *(optional)* how many independent values do you really have, when neighbouring values are correlated?

Sections 1–3 (and 5) are skills you will use in Exercise 2: describing and comparing groups with an interval on the difference, cleaning a time-ordered record, and honest uncertainties.

**Nothing here is assessed.**

## How to use this notebook today

**Core (about 1¾ hours):** Section 1 (a)–(c) and (e), Section 2, and Section 3(a). Everything marked *Optional* (1(d), 3(b), Sections 4 and 5, and the twenty tests) can wait, or be skipped.

**Exercise 1 is due on Friday.** Today is for the new material, but staff are happy to answer quick questions about the exercise.

**New to some of the Python?** Start with `week4_python_toolkit.ipynb` (15 minutes): masks, random numbers, `scipy.stats` and pandas tables. Everything below uses them.

**Behind on earlier weeks?** This lab doesn't depend on them, so it's a good week to catch up afterwards: `week3_derivatives_integrals.ipynb` (Sections 1 and 3) is the most useful for what comes next.

In [ ]:
import os

# Data: read from a local datasets/ folder if you've cloned/downloaded the
# repository, otherwise straight from GitHub (e.g. in Colab). Same file either way.
DATA_URL = "https://raw.githubusercontent.com/MatSmithAstro/phys7101_2026_resources/main/workshops/week4/datasets/"

def data_path(filename):
    local = os.path.join("datasets", filename)
    return local if os.path.exists(local) else DATA_URL + filename

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(2026)   # one generator, seeded, for the whole notebook

## <span style="color:#B5121B">1. How dense is the Earth? (Cavendish, 1798)</span>

Cavendish hung two small lead balls from a torsion balance, brought two large ones close, and measured the tiny twist. From 29 runs he found the mean density of the Earth. The values are in the order he measured them. After the 6th run he replaced the suspension wire with a stiffer one.

**(a) Look first.** Plot the 29 values in the order taken, and as a histogram. Anything odd?

In [ ]:
cav = pd.read_csv(data_path("cavendish_1798.csv"))
rho = cav["density_g_cm3"].to_numpy()
print(len(rho), "measurements")

In [ ]:
# TODO: two panels: rho against measurement number; a histogram of rho

**(b) Describe and quote.** Compute the mean, median, standard deviation and standard error of the mean. Recompute the mean and the median with the third value corrected to 5.88 (Stigler's reading of Cavendish's own arithmetic). Which statistic moves, and by how much? Quote the result as Lecture 5 asks, and compare it with the modern value, 5.513 g cm$^{-3}$, in units of your uncertainty.

In [ ]:
# TODO: mean, median, std (ddof=1), standard error
# TODO: the same with rho[2] = 5.88
# TODO: (5.513 - mean) / standard error

**Check:** mean 5.448, median 5.460 and standard error 0.041 g cm$^{-3}$ as printed; with 5.88 the mean becomes 5.482.

**(c) Did the new wire change anything?** Compare the first 6 measurements with the last 23 using Welch's t-test (`stats.ttest_ind(..., equal_var=False)`) and the Mann–Whitney rank test (`stats.mannwhitneyu`). State the null hypothesis, the two p-values, and what you conclude. Which test do you trust more here, and why?

In [ ]:
# TODO: before = rho[:6], after = rho[6:]; both tests; print the group means too

**Check:** before 5.312 (6 values), after 5.483 (23 values); Welch p = 0.22, Mann–Whitney p = 0.19.

**(d) Optional: a permutation test.** Lecture 5's nonparametric route. Shuffle the 29 values many times (`rng.permutation`), split each shuffle into 6 and 23, and record the difference of means. How often is it as large as the real one?

In [ ]:
# Optional

**(e) How big is the difference?** A p-value says whether a difference could be chance; it doesn't say how big it is. Estimate the difference of means (after − before) with a 95% **bootstrap** interval (Lecture 5): resample the 6 "before" values with replacement, and separately the 23 "after" values, compute the difference of their means, and repeat 10 000 times. The 2.5th and 97.5th percentiles of those differences are the interval. Does it include zero? Does it tell the same story as the p-values in (c), and what does it add?

<details><summary><b>Stuck? Open a hint</b> (try for 10 minutes first)</summary>

Use a generator of its own, so your interval doesn't depend on what you ran before:
```python
rng_boot = np.random.default_rng(5)
diffs = [rng_boot.choice(after, len(after)).mean() - rng_boot.choice(before, len(before)).mean() for _ in range(10_000)]
lo, hi = np.percentile(diffs, [2.5, 97.5])
```
Resample *each group separately*: the two groups keep their own sizes.

</details>

In [ ]:
# TODO: bootstrap the difference of means (after - before); 95% interval from the percentiles; plot the histogram

**Check:** the difference is 0.172 g cm$^{-3}$; with `default_rng(5)` the interval is −0.049 to +0.409 (another seed moves it by about 0.01).

## <span style="color:#B5121B">2. Glitches in a sequence</span>

Real records contain glitches: a reading taken at the wrong moment, a dropped sample, a value written down wrongly. In a time-ordered series the right question is not "is this value far from the mean?" but "is it far from its *neighbours*?", because the series may drift.

Below is a simulated record: a slow drift plus noise, with 12 glitches added at known places. Because you know where they are, you can measure how well a rule finds them.

**(a)** Plot the series. Then flag a value as a glitch if it is more than $k$ times the local spread from the **median of its 11 neighbours** (a rolling median: `pd.Series(y).rolling(11, center=True, min_periods=1).median()`). For the local spread, take each value's distance from its rolling median, $|y_i - \mathrm{median}_i|$, and find the rolling median of *those distances* over the same 11 neighbours (the rolling median absolute deviation, MAD). $1.4826 \times$ MAD is a robust stand-in for the standard deviation.

<details><summary><b>Stuck? Open a hint</b> (try for 10 minutes first)</summary>

Two rolling medians, one after the other:
```python
s = pd.Series(y)
med = s.rolling(11, center=True, min_periods=1).median()               # local level
mad = (s - med).abs().rolling(11, center=True, min_periods=1).median()  # local spread
flags = ((s - med).abs() > k * 1.4826 * mad).to_numpy()
```
Write it as a function of `y` and `k`, so (b) is a loop.

</details>

In [ ]:
n = 1500
t = np.arange(n) * 0.8                                     # one reading every 0.8 s
truth = 1.0 + 0.15 * np.sin(2 * np.pi * t / 400) + 0.0002 * t   # slow drift
rng_glitch = np.random.default_rng(42)                     # its own generator: everyone gets the same record
y = truth + 0.02 * rng_glitch.standard_normal(n)
glitch_at = np.sort(rng_glitch.choice(np.arange(20, n - 20), 12, replace=False))
y[glitch_at[:6]] *= 1.5                                    # six large glitches
y[glitch_at[6:]] *= 0.85                                   # six small ones
print("glitches at", glitch_at)

In [ ]:
# TODO: rolling median of 11 neighbours; rolling MAD; flag |y - median| > k * 1.4826 * MAD for k = 5
# TODO: plot the series, the rolling median and the flagged points

**(b)** For $k = 3, 4, 5, 6, 8$: how many of the 12 true glitches does the rule find, and how many good values does it flag wrongly? Which $k$ would you choose, and why? What happens to the small glitches?

In [ ]:
# TODO: for each k: found = flags & true-glitch mask; false alarms = flags & not glitch

**Check:** at $k = 5$ the rule finds 11 of the 12 glitches, with 14 false alarms.

In your logbook: what would you report about the glitches you removed, if this were your data? (Lecture 5: "plot the values in the order they were taken".)

## <span style="color:#B5121B">3. The speed of light: random and systematic (Michelson, 1879)</span>

Michelson measured the speed of light in air with a rotating mirror: five experiments of 20 runs each, on different days.

**(a)** Make a table with one row per experiment (`groupby`): the mean, standard deviation, standard error and number of runs. Plot all 100 values by experiment, with each experiment's mean and standard error marked.

In [ ]:
mic = pd.read_csv(data_path("michelson_1879.csv"))
mic.head()

In [ ]:
# TODO: table = mic.groupby("experiment")["speed_km_s"].agg(...)
# TODO: strip plot of every value by experiment, with mean ± SE

**Check:** experiment 1 has mean 299 909 km/s and standard error 23.5 km/s; experiment 4 has the lowest mean, 299 820.5 km/s.

**(b) Optional: random and systematic.** Combine all 100 runs into one result with its standard error. In air, the modern value is about 299 700 km/s (299 792.458 in vacuum, divided by the refractive index of air, 1.0003). How many standard errors away is Michelson's result? Are the five experiments consistent with one common value? (A χ² test of the five means against their weighted mean: the χ² of Lecture 3's fits, used as a test.)

In [ ]:
# TODO: overall mean and SE; (mean - 299700) / SE
# TODO: chi2 of the five means about their weighted mean; p from stats.chi2.sf(chi2, 4)

In your logbook: Michelson's *random* uncertainty is small. What does the gap to the modern value tell you, and would taking 1000 more runs have helped?

## <span style="color:#FD9029">4. Optional: propagating uncertainty into g</span>

A pendulum of length $L = 0.500 \pm 0.002$ m has a small-amplitude period $T = 1.419 \pm 0.005$ s, so $g = 4\pi^2 L / T^2$.

**(a) Linear rule (Lecture 5).** Compute $g$ and its uncertainty from the partial derivatives, assuming $L$ and $T$ are independent. Which input contributes more?

**(b) Monte Carlo.** Draw $10^5$ values of $L$ and $T$ from Gaussians (`rng.normal`), compute $g$ for each, and report the mean, standard deviation and the central 68% range. Plot the histogram. Do the two methods agree?

In [ ]:
# TODO: (a) g, dg/dL, dg/dT, sigma_g; each term's share
# TODO: (b) Monte Carlo with rng.normal; histogram; mean, std, 16th and 84th percentiles

**(c) When the linear rule fails.** Now suppose the period is much less certain, $T = 1.419 \pm 0.3$ s (a stopwatch started by hand on one swing). Repeat (a) and (b). Describe the Monte Carlo distribution of $g$: is "$g \pm \sigma$" still a fair summary?

In [ ]:
# TODO: repeat with sT = 0.4

## <span style="color:#FD9029">5. Optional: when neighbouring values are correlated</span>

$\sigma/\sqrt{N}$ assumes every value is independent. In a time-ordered record they often aren't: each value remembers the one before (Lecture 5). Below is a simulated record of 2000 values in which each is 0.8 times the previous one plus fresh noise, so neighbours are strongly correlated but the long-run mean is exactly 0.

**(a)** Compute the mean, the naive standard error $\sigma/\sqrt{N}$, and the lag-1 autocorrelation (the correlation of `x[:-1]` with `x[1:]`).

**(b) Block means.** Split the record into blocks of 50 consecutive values, take the mean of each block, and use the standard error of those 40 block means instead. Try blocks of 10, 20, 50 and 100. How much larger is the honest error than the naive one? For this kind of record theory says about $\sqrt{(1+a)/(1-a)} = 3$ times larger, with $a = 0.8$.

In [ ]:
rng_ar = np.random.default_rng(11)                          # its own generator
a, n_ar = 0.8, 2000
x = np.empty(n_ar); x[0] = rng_ar.standard_normal()
for i in range(1, n_ar):
    x[i] = a * x[i - 1] + rng_ar.standard_normal()            # each value remembers the previous one
plt.figure(figsize=(9, 2.5)); plt.plot(x, lw=0.6, color="#555656"); plt.xlabel("Index"); plt.ylabel("x"); plt.show()

In [ ]:
# TODO: (a) mean, naive SE, lag-1 autocorrelation
# TODO: (b) for block sizes 10, 20, 50, 100: SE from the block means; ratio to the naive SE

**Check:** naive standard error 0.037, lag-1 autocorrelation 0.80; blocks of 50 give a standard error about 2.5 times larger.

## <span style="color:#FD9029">Optional: twenty tests, no effect</span>

Lecture 5's last table warned that if you try many tests, some will give p < 0.05 by chance alone. How many? Simulate it: 20 pairs of groups of 15 values, all drawn from the *same* distribution, and a t-test on each. How many give $p < 0.05$? Repeat the whole experiment 1000 times: how often does at least one test come out "significant"? Compare with $1 - 0.95^{20}$.

In [ ]:
# Optional

## Summary

- **Look first:** plot values in the order taken; a histogram hides drifts and steps.
- **Mean or median:** the median barely notices one bad value; the mean does. Quote a result with its standard error and sensible precision.
- **Tests:** state the null hypothesis; a large p-value means "no evidence of a difference", not "no difference", especially with 6 values.
- **Glitches in a sequence:** compare each value with its neighbours, robustly; check the rule where you know the answer, and report what you removed.
- **Random and systematic:** averaging shrinks random error, never systematic error.
- **Size, not just significance:** a bootstrap interval on a difference says how big it could be; a p-value alone doesn't.
- **Propagation:** the linear rule is fine while the function is nearly straight over the uncertainty; otherwise use Monte Carlo and describe the distribution.
- **Correlated values:** neighbours that remember each other carry less information than their number suggests; block means give an honest error.

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis
</div>